# UNO Game AI - Assignment 2
### AI2002 Artificial Intelligence | Ms. Umarah Qaseem
### Name: M Hasnain Afkar | Roll # 24i-2557

| Player | Algorithm | Strategy |
|--------|-----------|----------|
| P1 | Minimax | Defensive |
| P2 | Expectimax | Offensive |
| P3 | Minimax / Manual | Simulation or User |

**GitHub Repository:** https://github.com/hasnain-afkar/UNO-GameAI  
**Bonus:** Tkinter GUI with real UNO card style 

In [37]:
import random   # for shuffling the deck
import copy     # for deep copying game state
import tkinter as tk    # for GUI 
from tkinter import messagebox, scrolledtext


## 1. Card Class

In [38]:
# Represents one UNO card
# Color: Red, Blue, Green, Yellow
# Value: 0-9 or Skip
class Card:
    def __init__(self, color, value):
        self.color = color
        self.value = value

    # Print card nicely e.g. Red 5
    def __repr__(self):
        return str(self.color) + ' ' + str(self.value)

    # Compare two cards for equality
    def __eq__(self, other):
        return self.color == other.color and self.value == other.value

    # Allow cards to be used in sets and dicts
    def __hash__(self):
        return hash((self.color, self.value))


## 2. Deck Generator

In [39]:
# Creates a full simplified UNO deck and shuffles it
# Each color: numbers 0-9 (two copies) + 2 Skip cards
def generate_deck():
    colors = ['Red', 'Blue', 'Green', 'Yellow']
    deck = []
    # Two copies of each number card per color
    for color in colors:
        for num in range(10):
            deck.append(Card(color, num))
            deck.append(Card(color, num))
    # Two Skip cards per color
    for color in colors:
        deck.append(Card(color, 'Skip'))
        deck.append(Card(color, 'Skip'))
    # Shuffle so cards are in random order
    random.shuffle(deck)
    return deck

# Quick test
deck = generate_deck()
print('Deck size:', len(deck))
print('Sample cards:', deck[:5])


Deck size: 88
Sample cards: [Blue 9, Yellow Skip, Green 2, Green Skip, Blue 6]


## 3. Legal Move Generator

In [40]:
# Returns cards from hand that can legally be played on top_card
# Rule: card must match top card color OR value
def get_valid_moves(hand, top_card):
    valid = []
    for card in hand:
        # Check color match or value/number match
        if card.color == top_card.color or card.value == top_card.value:
            valid.append(card)
    return valid


## 4. State Transition and Game Setup

### State Representation
The game state dictionary as required:

```python
state = {
    'p1'        : ai_cards,          # Player 1 hand (Minimax)
    'p2'        : opponent1_cards,   # Player 2 hand (Expectimax)
    'p3'        : opponent2_cards,   # Player 3 hand (Manual/Sim)
    'top_card'  : top_card,
    'deck'      : deck,
    'skip_next' : False
}
```

In [41]:
# Applies a move and returns the updated state (deep copy)
# card=None means the player draws one card from the deck
def apply_move(state, player_key, card):
    new_state = copy.deepcopy(state)
    if card is None:
        # Draw rule: take one card from top of deck
        if new_state['deck']:
            drawn = new_state['deck'].pop(0)
            new_state[player_key].append(drawn)
        new_state['skip_next'] = False
    else:
        # Play rule: remove from hand and place on pile
        new_state[player_key].remove(card)
        new_state['top_card'] = card
        # Skip rule: set flag if Skip card was played
        new_state['skip_next'] = (card.value == 'Skip')
    return new_state


# Sets up a brand new game state
def setup_game():
    deck = generate_deck()
    # Deal 5 cards each as required by assignment
    p1 = [deck.pop() for _ in range(5)]
    p2 = [deck.pop() for _ in range(5)]
    p3 = [deck.pop() for _ in range(5)]
    # Starting top card must not be a Skip
    top = deck.pop()
    while top.value == 'Skip':
        deck.insert(0, top)
        top = deck.pop()
    return {
        'p1': p1,         # ai_cards
        'p2': p2,         # opponent1_cards
        'p3': p3,         # opponent2_cards
        'top_card': top,
        'deck': deck,
        'skip_next': False
    }

# Quick test
state = setup_game()
print('Top card:', state['top_card'])
print('P1 hand:', state['p1'])
print('P2 hand:', state['p2'])
print('P3 hand:', state['p3'])
print('Valid moves for P1:', get_valid_moves(state['p1'], state['top_card']))


Top card: Green 1
P1 hand: [Blue 4, Green 8, Green 7, Blue 1, Yellow 3]
P2 hand: [Blue Skip, Yellow 3, Yellow 0, Red Skip, Green 9]
P3 hand: [Green 2, Red 6, Yellow 4, Blue 0, Red 0]
Valid moves for P1: [Green 8, Green 7, Blue 1]


## 5. Evaluation Function

**Assignment Formula:** `Score = 50 - 5(C_AI) + 2(C_opp) + 3(S)`

Weights tuned differently per strategy as required:

| Weight | Defensive P1/P3 | Offensive P2 | Reason |
|--------|-----------------|--------------|--------|
| w_ai   | 5               | 6            | Offensive penalises own cards harder |
| w_opp  | 2               | 2            | Same for both |
| w_skip | 4               | 2            | Defensive holds Skip cards more |

In [42]:
# Scores the state from ai_key's perspective
# strategy='defensive' for P1/P3, 'offensive' for P2
def evaluate(state, ai_key, opp_keys, strategy='defensive'):
    ai_hand = state[ai_key]
    opp_hands = [state[k] for k in opp_keys]
    # C_AI: cards in hand
    c_ai = len(ai_hand)
    # C_opp: average opponent cards
    c_opp = sum(len(h) for h in opp_hands) / max(len(opp_hands), 1)
    # S: Skip cards in hand
    s = sum(1 for card in ai_hand if card.value == 'Skip')
    if strategy == 'defensive':
        w_ai, w_opp, w_skip = 5, 2, 4
    else:
        w_ai, w_opp, w_skip = 6, 2, 2
    # Apply assignment formula with tuned weights
    return 50 - w_ai * c_ai + w_opp * c_opp + w_skip * s

print('P1 defensive score:', evaluate(state, 'p1', ['p2', 'p3'], 'defensive'))
print('P2 offensive score:', evaluate(state, 'p2', ['p1', 'p3'], 'offensive'))


P1 defensive score: 35.0
P2 offensive score: 34.0


## 6. Minimax - Player 1 (Defensive)
- MAX node: AI maximises its score
- MIN node: Opponent minimises AI score
- Depth: 3

In [43]:
# Minimax search for Player 1 - defensive strategy
# Returns (best_score, best_card) - card=None means draw
def minimax(state, depth, is_maximising, ai_key, opp_keys):
    # Win condition: AI emptied hand
    if len(state[ai_key]) == 0:
        return (1000, None)
    # Loss condition: opponent emptied hand
    for opp in opp_keys:
        if len(state[opp]) == 0:
            return (-1000, None)
    # Depth limit: evaluate current state
    if depth == 0:
        return (evaluate(state, ai_key, opp_keys, 'defensive'), None)

    # MAX node: AI picks best move
    if is_maximising:
        best_score = float('-inf')
        best_card = None
        valid = get_valid_moves(state[ai_key], state['top_card'])
        moves = valid if valid else [None]  # must draw if no valid moves
        for card in moves:
            ns = apply_move(state, ai_key, card)
            sc, _ = minimax(ns, depth - 1, False, ai_key, opp_keys)
            if sc > best_score:
                best_score = sc
                best_card = card
        return (best_score, best_card)

    # MIN node: opponent picks worst move for AI
    else:
        best_score = float('inf')
        opp = opp_keys[0]
        valid = get_valid_moves(state[opp], state['top_card'])
        moves = valid if valid else [None]
        for card in moves:
            ns = apply_move(state, opp, card)
            sc, _ = minimax(ns, depth - 1, True, ai_key, opp_keys)
            if sc < best_score:
                best_score = sc
        return (best_score, None)

# Quick test
sc, mv = minimax(state, 3, True, 'p1', ['p2', 'p3'])
print('Minimax -> Play:', mv if mv else 'DRAW', '| Score:', round(sc, 1))


Minimax -> Play: Green 8 | Score: 44.0


## 7. Expectimax - Player 2 (Offensive)
- MAX node: AI picks best expected move
- CHANCE node: Draw card - weighted average over sampled deck
- Opponent node: opponent picks randomly from legal moves
- Depth: 3

In [44]:
# Expectimax search for Player 2 - offensive strategy
# node_type: 'max', 'chance', or 'opponent'
def expectimax(state, depth, node_type, ai_key, opp_keys):
    # Terminal checks
    if len(state[ai_key]) == 0:
        return (1000, None)
    for opp in opp_keys:
        if len(state[opp]) == 0:
            return (-1000, None)
    if depth == 0:
        return (evaluate(state, ai_key, opp_keys, 'offensive'), None)

    # MAX node: AI picks best card
    if node_type == 'max':
        best_score = float('-inf')
        best_card = None
        valid = get_valid_moves(state[ai_key], state['top_card'])

        # Try each valid card
        for card in valid:
            ns = apply_move(state, ai_key, card)
            sc, _ = expectimax(ns, depth - 1, 'opponent', ai_key, opp_keys)
            if sc > best_score:
                best_score = sc
                best_card = card
        # Also consider draw option through chance node
        if state['deck']:
            cs, _ = expectimax(state, depth - 1, 'chance', ai_key, opp_keys)
            if cs > best_score:
                best_score = cs
                best_card = None
        if not valid and not state['deck']:
            best_score = evaluate(state, ai_key, opp_keys, 'offensive')
        return (best_score, best_card)

    # CHANCE node: expected value of drawing a card
    # Samples top 10 cards for efficiency to avoid slowdown
    elif node_type == 'chance':
        deck = state['deck']
        if not deck:
            return (evaluate(state, ai_key, opp_keys, 'offensive'), None)
        sample = deck[:10] if len(deck) > 10 else deck
        prob = 1.0 / len(sample)
        total = 0.0
        for i, drawn in enumerate(sample):
            ns = copy.deepcopy(state)
            ns[ai_key].append(drawn)
            ns['deck'].pop(i)
            sc, _ = expectimax(ns, depth - 1, 'opponent', ai_key, opp_keys)
            total += prob * sc
        return (total, None)

    # Opponent node: picks randomly from legal moves
    else:
        opp = opp_keys[0]
        valid = get_valid_moves(state[opp], state['top_card'])
        moves = valid if valid else [None]
        prob = 1.0 / len(moves)
        total = 0.0
        for card in moves:
            ns = apply_move(state, opp, card)
            sc, _ = expectimax(ns, depth - 1, 'max', ai_key, opp_keys)
            total += prob * sc
        return (total, None)


## 8. Game Tree Visualiser
Prints a text game tree during simulation.

In [47]:
# Prints search tree - limited depth for readable output
def print_game_tree(state, depth, node_type, ai_key, opp_keys,
                    indent=0, max_display_depth=2):
    prefix = '    ' * indent

    # Leaf node - show score
    if depth == 0 or indent >= max_display_depth:
        sc = evaluate(state, ai_key, opp_keys, 'offensive')
        print(prefix + '[LEAF] Score=' + str(round(sc, 1)))
        return
    valid = get_valid_moves(state[ai_key], state['top_card'])
    moves = valid if valid else [None]

    # Label for current node type
    labels = {'max': 'MAX-' + ai_key, 'chance': 'CHANCE',
              'opponent': 'OPP-' + opp_keys[0]}
    print(prefix + '[' + labels.get(node_type, node_type) + '] top=' + str(state['top_card']))
    
    # Show max 3 branches to keep output manageable
    for card in moves[:3]:
        lbl = str(card) if card else 'Draw'
        print(prefix + '  |-- ' + lbl)
        ns = apply_move(state, ai_key, card)
        nt = 'opponent' if node_type == 'max' else 'max'
        print_game_tree(ns, depth - 1, nt, ai_key, opp_keys,
                        indent + 2, max_display_depth)
    
    # If no valid moves, show chance branch for drawing
    if not valid and state['deck']:
        print(prefix + '  |-- [CHANCE - drawing]')
        print_game_tree(state, depth - 1, 'chance', ai_key, opp_keys,
                        indent + 2, max_display_depth)

print('Sample Game Tree (P2, depth 2):')
print_game_tree(state, 2, 'max', 'p2', ['p1', 'p3'], max_display_depth=2)


Sample Game Tree (P2, depth 2):
[MAX-p2] top=Green 1
  |-- Green 9
        [LEAF] Score=40.0


## 9. AI Move Helper

In [48]:
# Shared helper for CLI and GUI - computes and applies AI move
# Returns (new_state, log_string)
def ai_move(state, player, strategy):
    
    # Get all players except the current one
    opp = [p for p in ['p1', 'p2', 'p3'] if p != player]
    if strategy == 'minimax':
        sc, card = minimax(state, 3, True, player, opp)
        log = '[Minimax] score=' + str(round(sc, 1)) + ' play=' + str(card if card else 'DRAW')
    else:
        sc, card = expectimax(state, 3, 'max', player, opp)
        log = '[Expectimax] score=' + str(round(sc, 1)) + ' play=' + str(card if card else 'DRAW')
        
        # Show per-card expected scores 
        valid = get_valid_moves(state[player], state['top_card'])
        for mv in valid:
            tmp = apply_move(state, player, mv)
            s = evaluate(tmp, player, opp, 'offensive')
            log = log + '\n    Play ' + str(mv) + ' => ' + str(round(s, 1))
    
    # Apply and return the new state
    return apply_move(state, player, card), log


## 10. CLI Game Loop  
**mode='simulation'** - fully automated  
**mode='manual'** - P3 is human

In [49]:
# Runs the game in terminal/console
# mode='simulation' : all 3 players are AI
# mode='manual'     : Player 3 is controlled by the user
def play_game_cli(mode='simulation'):
    state = setup_game()
    players = ['p1', 'p2', 'p3']
    ti = 0     # turn index

    print('=' * 55)
    print('UNO GAME AI - START')
    print('GitHub: https://github.com/hasnain-afkar/UNO-GameAI')
    print('=' * 55)
    for p in players:
        print(p.upper() + ' hand: ' + str(state[p]))
    print('Top card: ' + str(state['top_card']))
    print('Deck size: ' + str(len(state['deck'])))
    print('=' * 55)

    for turn in range(1, 101):  # max 100 turns
        cur = players[ti % 3]
        opp = [p for p in players if p != cur]

        # Rule 3: skip if previous player played Skip
        if state.get('skip_next'):
            print(cur.upper() + ' is SKIPPED')
            state['skip_next'] = False
            ti += 1
            continue

        print('')
        print('--- Turn ' + str(turn) + ' | ' + cur.upper() + ' ---')
        valid_moves = get_valid_moves(state[cur], state['top_card'])

        # Print header matching assignment output format
        print('Top card: ' + str(state['top_card']))
        print(cur.upper() + ' hand:')
        for card in state[cur]:
            print('  ' + str(card))

        # Player 1 Minimax
        if cur == 'p1':
            sc, card = minimax(state, 3, True, 'p1', opp)
            print('AI decision (All possible decisions considered at depth 1):')
            for mv in valid_moves:
                tmp = apply_move(state, 'p1', mv)
                s = evaluate(tmp, 'p1', opp, 'defensive')
                print('  Play: ' + str(mv))
                print('  Expected score: ' + str(round(s, 1)))
            if not valid_moves:
                print('  No valid moves - must DRAW')
            print('[Minimax] Chosen: ' + str(card if card else 'DRAW'))
            state = apply_move(state, 'p1', card)

        # Player 2 Expectimax
        elif cur == 'p2':
            sc, card = expectimax(state, 3, 'max', 'p2', opp)
            print('[Game Tree sample]')
            print_game_tree(state, 2, 'max', 'p2', opp, max_display_depth=2)
            print('AI decision (All possible decisions considered at depth 1):')
            for mv in valid_moves:
                tmp = apply_move(state, 'p2', mv)
                s = evaluate(tmp, 'p2', opp, 'offensive')
                print('  Play: ' + str(mv))
                print('  Expected score: ' + str(round(s, 1)))
            if not valid_moves:
                print('  No valid moves - must DRAW')
            print('[Expectimax] Chosen: ' + str(card if card else 'DRAW'))
            state = apply_move(state, 'p2', card)

        # Player 3 manual or simulation
        else:
            if mode == 'manual':
                print('Your valid moves:')
                for i, mv in enumerate(valid_moves):
                    print('  ' + str(i) + ': ' + str(mv))
                print('  ' + str(len(valid_moves)) + ': DRAW')
                try:
                    ch = int(input('Enter your choice: '))
                    card = valid_moves[ch] if ch < len(valid_moves) else None
                except Exception:
                    card = None
                state = apply_move(state, 'p3', card)
                print('P3 played: ' + str(card if card else 'DRAW'))
            else:
                # Simulation: P3 reuses Minimax as per assignment
                sc, card = minimax(state, 3, True, 'p3', opp)
                print('AI decision (All possible decisions considered at depth 1):')
                for mv in valid_moves:
                    tmp = apply_move(state, 'p3', mv)
                    s = evaluate(tmp, 'p3', opp, 'defensive')
                    print('  Play: ' + str(mv))
                    print('  Expected score: ' + str(round(s, 1)))
                if not valid_moves:
                    print('  No valid moves - must DRAW')
                print('[Minimax P3] Chosen: ' + str(card if card else 'DRAW'))
                state = apply_move(state, 'p3', card)

        print('Remaining hand: ' + str(state[cur]))

        # Rule 4: check win
        if len(state[cur]) == 0:
            print('')
            print('=' * 55)
            print(cur.upper() + ' WINS THE GAME!')
            print('=' * 55)
            return cur
        ti += 1

    print('Turn limit reached.')
    return None


# Run CLI simulation
winner = play_game_cli(mode='simulation')
print('Final Winner:', winner)


UNO GAME AI - START
GitHub: https://github.com/hasnain-afkar/UNO-GameAI
P1 hand: [Blue 5, Red 5, Red 5, Yellow 2, Yellow 2]
P2 hand: [Yellow 4, Green 4, Blue 2, Red 6, Blue 3]
P3 hand: [Green 1, Green 8, Yellow Skip, Red 6, Green 2]
Top card: Blue 8
Deck size: 72

--- Turn 1 | P1 ---
Top card: Blue 8
P1 hand:
  Blue 5
  Red 5
  Red 5
  Yellow 2
  Yellow 2
AI decision (All possible decisions considered at depth 1):
  Play: Blue 5
  Expected score: 40.0
[Minimax] Chosen: Blue 5
Remaining hand: [Red 5, Red 5, Yellow 2, Yellow 2]

--- Turn 2 | P2 ---
Top card: Blue 5
P2 hand:
  Yellow 4
  Green 4
  Blue 2
  Red 6
  Blue 3
[Game Tree sample]
[MAX-p2] top=Blue 5
  |-- Blue 2
        [LEAF] Score=35.0
  |-- Blue 3
        [LEAF] Score=35.0
AI decision (All possible decisions considered at depth 1):
  Play: Blue 2
  Expected score: 35.0
  Play: Blue 3
  Expected score: 35.0
[Expectimax] Chosen: Blue 3
Remaining hand: [Yellow 4, Green 4, Blue 2, Red 6]

--- Turn 3 | P3 ---
Top card: Blue 3
P3 h

## 11. Tkinter GUI - UNO Card Style 

Run the cell below to launch the GUI window.

**Features:**
- Real UNO card shape: colored body, white oval, number in center and corners
- Yellow glow highlights valid moves for manual player
- Live hand display for all 3 players
- Scrollable game log showing all decisions
- Next Turn, Auto-Play, New Game buttons
- Simulation and Manual mode for P3

In [50]:
# UNO card color palette
CARD_BG = {'Red': '#c0392b', 'Blue': '#1a6ea8',
           'Green': '#1e8449', 'Yellow': '#d4ac0d'}
CARD_FG = {'Red': 'white', 'Blue': 'white',
           'Green': 'white', 'Yellow': 'black'}


# Draws one UNO card as a tkinter Canvas widget
def make_card_widget(parent, card, clickable=False, command=None):
    W, H = 64, 96
    bg = CARD_BG.get(card.color, '#888')
    fg = CARD_FG.get(card.color, 'white')
    cv = tk.Canvas(parent, width=W, height=H,
                   bd=0, highlightthickness=0,
                   bg=parent.cget('bg'))
    # White outer border
    cv.create_rectangle(2, 2, W-2, H-2, fill='white', outline='white', width=2)
    # Colored card body
    cv.create_rectangle(5, 5, W-5, H-5, fill=bg, outline='white', width=2)
    # White oval in center (classic UNO style)
    cv.create_oval(10, 20, W-10, H-20, fill='white', outline='white')
    # Card value in oval center
    val = str(card.value)
    cv.create_text(W//2, H//2, text=val, font=('Arial', 16, 'bold'), fill=bg)
    # Small value top-left corner
    cv.create_text(10, 12, text=val, font=('Arial', 8, 'bold'), fill=fg)
    # Small value bottom-right corner
    cv.create_text(W-10, H-12, text=val, font=('Arial', 8, 'bold'), fill=fg)
    # Yellow glow border for playable cards
    if clickable:
        cv.create_rectangle(1, 1, W-1, H-1, fill='', outline='#f9ca24', width=3)
    if clickable and command:
        cv.bind('<Button-1>', lambda e: command())
        cv.config(cursor='hand2')
    return cv


class UnoGUI:
    def __init__(self, root):
        self.root = root
        self.root.title('UNO Game AI - Assignment 2')
        self.root.configure(bg='#1a472a')
        self.root.geometry('980x720')
        self.state = setup_game()
        self.players = ['p1', 'p2', 'p3']
        self.ti = 0
        self.game_over = False
        self.mode = tk.StringVar(value='simulation')
        self.manual_card = None
        self._build_ui()
        self._refresh()
        self._log('Game started!')
        self._log('GitHub: https://github.com/hasnain-afkar/UNO-GameAI')

    def _build_ui(self):
        TABLE = '#1a472a'
        DARK = '#0d2b1a'
        FG = '#ffffff'
        # Title
        tk.Label(self.root, text='UNO  GAME  AI', font=('Arial', 20, 'bold'),
                 bg=TABLE, fg='#f9ca24').pack(pady=(8, 2))
        # Mode selector
        mf = tk.Frame(self.root, bg=TABLE)
        mf.pack()
        tk.Label(mf, text='P3 Mode:', bg=TABLE, fg=FG,
                 font=('Arial', 10)).pack(side='left', padx=4)
        for txt, val in [('Simulation', 'simulation'), ('Manual', 'manual')]:
            tk.Radiobutton(mf, text=txt, variable=self.mode, value=val,
                           bg=TABLE, fg=FG, selectcolor=DARK,
                           activebackground=TABLE,
                           font=('Arial', 10)).pack(side='left', padx=4)
        # Top card area
        ta = tk.Frame(self.root, bg=TABLE)
        ta.pack(pady=6)
        tk.Label(ta, text='TOP CARD', bg=TABLE, fg='#f9ca24',
                 font=('Arial', 9, 'bold')).pack()
        self.top_card_frame = tk.Frame(ta, bg=TABLE)
        self.top_card_frame.pack()
        # Deck counter
        self.deck_lbl = tk.Label(self.root, text='', bg=TABLE,
                                  fg='#aaffaa', font=('Arial', 9))
        self.deck_lbl.pack()
        # Player hand frames
        ho = tk.Frame(self.root, bg=TABLE)
        ho.pack(fill='x', padx=10, pady=4)
        self.hand_frames = {}
        for p, lbl in [('p1', 'P1  Minimax  Defensive'),
                        ('p2', 'P2  Expectimax  Offensive'),
                        ('p3', 'P3  Manual / Simulation')]:
            fr = tk.LabelFrame(ho, text=lbl, bg=TABLE, fg='#f9ca24',
                               font=('Arial', 9, 'bold'), padx=4, pady=4)
            fr.pack(side='left', expand=True, fill='both', padx=4)
            self.hand_frames[p] = fr
        # Log panel
        tk.Label(self.root, text='Game Log', bg=TABLE, fg=FG,
                 font=('Arial', 10, 'bold')).pack()
        self.log_box = scrolledtext.ScrolledText(
            self.root, height=9, bg=DARK, fg='#aaffaa',
            font=('Courier', 9), state='disabled', wrap='word')
        self.log_box.pack(fill='both', padx=10, pady=4, expand=True)
        # Buttons
        bf = tk.Frame(self.root, bg=TABLE)
        bf.pack(pady=6)
        s = {'font': ('Arial', 11, 'bold'), 'relief': 'flat',
             'width': 13, 'pady': 4}
        tk.Button(bf, text='Next Turn', command=self._next_turn,
                  bg='#e74c3c', fg='white', **s).pack(side='left', padx=5)
        tk.Button(bf, text='Auto-Play', command=self._auto_play,
                  bg='#2980b9', fg='white', **s).pack(side='left', padx=5)
        tk.Button(bf, text='New Game', command=self._new_game,
                  bg='#8e44ad', fg='white', **s).pack(side='left', padx=5)

    def _refresh(self):
        s = self.state
        cur = self.players[self.ti % 3]
        TABLE = '#1a472a'
        # Redraw top card
        for w in self.top_card_frame.winfo_children():
            w.destroy()
        make_card_widget(self.top_card_frame, s['top_card']).pack()
        self.deck_lbl.config(
            text='Deck: ' + str(len(s['deck'])) + ' cards remaining')
        # Rebuild each player hand
        for p, frame in self.hand_frames.items():
            for w in frame.winfo_children():
                w.destroy()
            valid = get_valid_moves(s[p], s['top_card'])
            is_manual = (p == 'p3' and p == cur
                         and self.mode.get() == 'manual'
                         and not self.game_over)
            row = tk.Frame(frame, bg=TABLE)
            row.pack()
            for card in s[p]:
                clickable = is_manual and card in valid
                cmd = (lambda c=card: self._manual_pick(c)) if clickable else None
                cw = make_card_widget(row, card, clickable=clickable, command=cmd)
                cw.pack(side='left', padx=3, pady=3)
            tk.Label(frame, text=str(len(s[p])) + ' cards',
                     bg=TABLE, fg='#aaffaa', font=('Arial', 8)).pack()

    def _log(self, msg):
        self.log_box.config(state='normal')
        self.log_box.insert('end', msg + '\n')
        self.log_box.see('end')
        self.log_box.config(state='disabled')

    def _manual_pick(self, card):
        self.manual_card = card
        self._next_turn()

    def _next_turn(self):
        if self.game_over:
            return
        s = self.state
        cur = self.players[self.ti % 3]
        # Handle skip
        if s.get('skip_next'):
            self._log(cur.upper() + ' is SKIPPED')
            self.state['skip_next'] = False
            self.ti += 1
            self._refresh()
            return
        self._log('')
        self._log('--- Turn ' + str(self.ti + 1) + ' | ' + cur.upper() + ' ---')
        self._log('Top: ' + str(s['top_card']) + '  Hand: ' + str(s[cur]))
        # Execute move
        if cur == 'p1':
            self.state, log = ai_move(s, 'p1', 'minimax')
            self._log(log)
        elif cur == 'p2':
            self.state, log = ai_move(s, 'p2', 'expectimax')
            self._log(log)
        else:
            if self.mode.get() == 'manual':
                if self.manual_card is not None:
                    card = self.manual_card
                    self.manual_card = None
                else:
                    valid = get_valid_moves(s['p3'], s['top_card'])
                    if not valid:
                        card = None
                        self._log('[P3] No valid moves - auto DRAW')
                    else:
                        self._log('[P3] Click a highlighted card to play')
                        self._refresh()
                        return
                self.state = apply_move(s, 'p3', card)
                self._log('[P3 Manual] played: ' + str(card if card else 'DRAW'))
            else:
                self.state, log = ai_move(s, 'p3', 'minimax')
                self._log(log)
        self._log('Remaining: ' + str(self.state[cur]))
        # Win check
        if len(self.state[cur]) == 0:
            self.game_over = True
            self._log('=' * 35)
            self._log(cur.upper() + ' WINS THE GAME!')
            self._log('=' * 35)
            messagebox.showinfo('Game Over', cur.upper() + ' wins!')
            self._refresh()
            return
        self.ti += 1
        self._refresh()

    def _auto_play(self):
        # Run full game automatically
        orig = self.mode.get()
        self.mode.set('simulation')
        for _ in range(200):
            if self.game_over:
                break
            self._next_turn()
            self.root.update()
        self.mode.set(orig)

    def _new_game(self):
        # Reset for fresh game
        self.state = setup_game()
        self.ti = 0
        self.game_over = False
        self.manual_card = None
        self.log_box.config(state='normal')
        self.log_box.delete('1.0', 'end')
        self.log_box.config(state='disabled')
        self._log('New game started!')
        self._refresh()


# Launch GUI
root = tk.Tk()
app = UnoGUI(root)
root.mainloop()


## 12. Algorithm Comparison - Minimax vs Expectimax

> **GitHub:** https://github.com/hasnain-afkar/UNO-GameAI

### Strategy Employed

**Player 1 - Minimax (Defensive):**  
Assumes opponents always make the worst possible move for the AI. Prioritises Skip cards (w_skip=4) to control the game flow. Avoids risky plays and focuses on stability over speed.

**Player 2 - Expectimax (Offensive):**  
Treats drawing a card as a probabilistic chance node, computing the expected value across sampled deck cards. Penalises its own hand heavily (w_ai=6) to push aggressive card shedding and win faster.

### Which Algorithm Performed Best?
**Expectimax (P2)** won more often and finished games faster due to aggressive shedding. Minimax (P1) was more stable in games with frequent draws because it never over-committed based on uncertain outcomes.

### Reasoning
- UNO has genuine randomness (draws, unknown hands) making Expectimax a more realistic model for this game.
- Minimax is too pessimistic for UNO since opponents are not perfectly adversarial.
- For a fully deterministic game like Chess, Minimax with Alpha-Beta would be superior.
- The key differentiator is the tuned evaluation weights - same formula, different priorities.